# Tomás Solano, InZenFenix, IIT414W, 20-04-2026

# CELL 1. Reproducibility, Libraries and Cache

In [1]:
# ── Cell 1: Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, random
import numpy as np
import warnings
import platform

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')
print("\nSystem: " + platform.system())
print("Distro: " + platform.release())

Python  : 3.14.4
NumPy   : 2.2.3
Seed    : 414

System: Linux
Distro: 6.19.12-1-cachyos


In [ ]:
# ── Cell 2: Dependency Guard ───────────────────────────────────────────────
# Ensures all required packages are installed in the active kernel.
# Safe to re-run: pip will skip already-installed packages.

import importlib, subprocess, sys

_REQUIRED = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'fastf1': 'fastf1',
}

_missing = []
for _mod, _pip in _REQUIRED.items():
    try:
        importlib.import_module(_mod)
    except ModuleNotFoundError:
        _missing.append(_pip)

if _missing:
    print(f'Installing missing packages: {_missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + _missing)
    print('Done. Packages installed successfully.')
else:
    print('All required packages already installed ✓')

# ── Library Imports ───────────────────────────────────────────────
import os                        # Working directory checks
import subprocess                # Git command checks
import importlib                 # Runtime dependency checks
import numpy as np               # Numeric support
import pandas as pd              # Tables and diagnostics
import fastf1                    # Formula 1 data access
from sklearn.linear_model import LogisticRegression # Model
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ( #Scores
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


print(f'fastf1  : {fastf1.__version__}')
print(f'pandas  : {pd.__version__}')

All required packages already installed ✓
fastf1  : 3.8.1
pandas  : 2.2.3


## IMPORTANT INFORMATION

- We are trying to predict if a Driver ends top 10 by the end of the race.
- Last time we created a heuristic model with the racer's team and compare it with simple models.
- This time we are creating 5 models, 2 baselines and 3 "improved" models.
- The main metric will be macro F1 (since our target is a classification)

## CELL 2.0 -- OBTAINING DATA

- We are gonna use 22 rounds for 2024 and 5 for 2025 (train and test respectively)

In [39]:
def get_session_data(year, rounds):
    data_frames = []

    for r in range(1, rounds + 1):
        try:
            session = fastf1.get_session(year, r, 'Q')
            session.load(telemetry=False, weather=False, messages=False)

            res = session.results.copy()

            df_round = res[['FullName', 'Position', 'TeamName', 'Abbreviation']].copy()

            # --- Contextual Features ---
            df_round['Round'] = r
            df_round['Year'] = year
            df_round['Circuit'] = session.event.get('EventName', f'Round_{r}')

            # --- Best Lap Time: use Q1/Q2/Q3, take the best non-null ---
            q_cols = [c for c in ['Q1', 'Q2', 'Q3'] if c in res.columns]
            if q_cols:
                best_laps = res[q_cols].apply(
                    lambda col: col.dt.total_seconds(), axis=0
                )
                df_round['BestLapTime_s'] = best_laps.min(axis=1)  # best Q segment per driver
            else:
                df_round['BestLapTime_s'] = float('nan')

            # --- Lap Time Delta vs fastest ---
            fastest_lap = df_round['BestLapTime_s'].min()
            df_round['LapTimeDelta'] = df_round['BestLapTime_s'] - fastest_lap
            df_round['LapTimeDelta'] = df_round['LapTimeDelta'].fillna(5.0)  # penalize DNS/DNQ

            # --- Q segment reached (ordinal: 1, 2, or 3) ---
            def q_stage_reached(row):
                if pd.notna(res.loc[row.name, 'Q3']) if 'Q3' in res.columns else False:
                    return 3
                elif pd.notna(res.loc[row.name, 'Q2']) if 'Q2' in res.columns else False:
                    return 2
                return 1
            df_round['QStageReached'] = df_round.apply(q_stage_reached, axis=1)

            # --- Grid Position (same as Position in Q, but explicit) ---
            df_round['GridPosition'] = pd.to_numeric(df_round['Position'], errors='coerce')

            # --- Target ---
            df_round['target_top10'] = (df_round['GridPosition'] <= 10).astype(int)

            data_frames.append(df_round)
            print(f"✓ Loaded {year} Round {r}: {df_round['Circuit'].iloc[0]}")

        except Exception as e:
            print(f"✗ Skipping {year} Round {r}: {e}")

    if not data_frames:
        print("CRITICAL: No data loaded.")
        return pd.DataFrame()

    return pd.concat(data_frames, ignore_index=True)

def add_ml_features(df):
    """Adds the features for the models"""
    df = df.sort_values(['FullName', 'Year', 'Round'])
    
    # 1. Driver Momentum: Rolling Top 10 rate (Last 5 races)
    # We shift(1) so the model doesn't know the result of the CURRENT race 
    df['driver_form'] = df.groupby('FullName')['target_top10'].transform(
        lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
    )
    
    # 2. Team Strength: Expanding average position (Leakage prevention)
    df['team_strength'] = df.groupby('TeamName')['Position'].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    
    # 3. Lap Time Consistency: Driver's avg delta this season (Leakage prevention)
    df['avg_delta_season'] = df.groupby(['FullName', 'Year'])['LapTimeDelta'].transform(
        lambda x: x.shift(1).expanding().mean()
    )

    # Clean up NaNs from the first races of the season
    df = df.fillna({
        'driver_form': 0.5, 
        'team_strength': 10.5,
        'avg_delta_season': 1.0
    })
    
    return df


# 1. DATA ACQUISITION
# Training on 2024, Testing on early 2025 (Temporal Split)
print("Downloading F1 Data...")

train_raw = get_session_data(2024, 22)
test_raw = get_session_data(2025, 5)
full_df = pd.concat([train_raw, test_raw], ignore_index=True)
full_df = add_ml_features(full_df)

display(full_df)

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO

✓ Loaded 2024 Round 1: Bahrain Grand Prix


core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '14', '81', '4', '63', '44', '22', '18', '38', '23', '20', '3', '27', '77', '31', '10', '2', '24']
core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✓ Loaded 2024 Round 2: Saudi Arabian Grand Prix


core           INFO 	Finished loading data for 19 drivers: ['1', '55', '11', '4', '16', '81', '63', '22', '18', '14', '44', '23', '77', '20', '31', '27', '10', '3', '24']
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

core           INFO 	Finished loading data for 20 drivers: ['81', '63', '16', '12', '10', '4', '1', '55', '44', '22', '7', '6', '14', '31', '23', '27', '30', '5', '18', '87']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✓ Loaded 2025 Round 4: Bahrain Grand Prix


core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core           INFO 	Finished loading data for 20 drivers: ['1', '81', '63', '16', '12', '55', '44', '22', '10', '4', '23', '30', '14', '6', '87', '18', '7', '27', '31', '5']


✓ Loaded 2025 Round 5: Saudi Arabian Grand Prix


,FullName,Position,TeamName,Abbreviation,Round,Year,Circuit,BestLapTime_s,LapTimeDelta,QStageReached,GridPosition,target_top10,driver_form,team_strength,avg_delta_season
12,Alexander Albon,13.0,Williams,ALB,1,2024,Bahrain Grand Prix,90.221,1.056,2,13.0,0,0.5,10.500000,1.000000
31,Alexander Albon,12.0,Williams,ALB,2,2024,Saudi Arabian Grand Prix,88.980,1.508,2,12.0,0,0.0,13.000000,1.056000
51,Alexander Albon,12.0,Williams,ALB,3,2024,Australian Grand Prix,77.130,1.215,2,12.0,0,0.0,12.500000,1.282000
72,Alexander Albon,14.0,Williams,ALB,4,2024,Japanese Grand Prix,89.714,1.517,2,14.0,0,0.0,12.333333,1.259667
92,Alexander Albon,14.0,Williams,ALB,5,2024,Chinese Grand Prix,95.241,1.581,2,14.0,0,0.0,12.750000,1.324000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,Yuki Tsunoda,5.0,Racing Bulls,TSU,1,2025,Australian Grand Prix,75.670,0.574,3,5.0,1,0.6,11.750000,1.000000
467,Yuki Tsunoda,9.0,Racing Bulls,TSU,2,2025,Chinese Grand Prix,91.238,0.597,3,9.0,1,0.6,11.000000,0.574000
493,Yuki Tsunoda,15.0,Red Bull Racing,TSU,3,2025,Japanese Grand Prix,87.967,0.984,2,15.0,0,0.8,6.294118,0.585500
508,Yuki Tsunoda,10.0,Red Bull Racing,TSU,4,2025,Bahrain Grand Prix,91.228,1.387,3,10.0,1,0.8,6.461538,0.718333


## CELL 2.1 -- HEUSTERIC BASELINE

- Baselines only use one feature to work (position, team, etc.), however the temporal set is the same for both the baseline and the models

### Heuristic 1: "**Your predicted grid position is the average of your last 3 qualifying sessions**"

- Leakage Prevention: Ensures no leakage by shifting BEFORE calculating the rolling mean.
- Type of baseline: Form (Looking at recent history). 

In [37]:
# ---- HEURISTIC 1 MODEL -----

# 1. HEURISTIC FEATURE ENGINEERING
def apply_heuristic(df):
    # Sort by time to prevent leakage
    df = df.sort_values(['FullName', 'Year', 'Round'])
    
    # Create the 'Shift' (The Firewall)
    # This moves the result of the previous race to the current row
    df['prev_pos'] = df.groupby('FullName')['Position'].shift(1)
    
    # Calculate 3-race rolling average of PREVIOUS performances
    df['h_avg_pos'] = df.groupby('FullName')['prev_pos'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    
    # Fill gaps for new drivers with the grid mid-point
    df['h_avg_pos'] = df['h_avg_pos'].fillna(10.5)
    
    # TARGET: Top-10 Finish (1 if Position <= 10, else 0)
    df['target_top10'] = (df['Position'] <= 10).astype(int)
    
    # HEURISTIC PREDICTION: Predicted to be in Top 10 if average pos <= 10
    df['pred_top10'] = (df['h_avg_pos'] <= 10).astype(int)
    
    # SCORING PROBABILITY: Inverted average for ROC-AUC/PR (Lower rank = higher 'score')
    df['h_score'] = 1 / df['h_avg_pos']
    
    return df

processed_df = apply_heuristic(full_df)

# 2. TEMPORAL EVALUATION
test_final = processed_df[processed_df['Year'] == 2025].copy()

train_final = processed_df[processed_df['Year'] == 2024].copy()

def get_scores(y_true, y_pred, y_score):
    return {
        "F1 (Macro)": f1_score(y_true, y_pred, average='macro'),
        "Accuracy": accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "PR-AUC (Avg Precision)": average_precision_score(y_true, y_score)
    }

# Score the Test Set 
test_h_results = get_scores(test_final['target_top10'], 
                            test_final['pred_top10'], 
                            test_final['h_score'])

# Score the Train Set 
train_h_results = get_scores(train_final['target_top10'], 
                     train_final['pred_top10'], 
                     train_final['h_score'])


# 4. OUTPUT RESULTS
print("\n--- First Heuristic Baseline: 3-Race Rolling Average ---")

print("\nTRAINING METRIC")
for metric, value in train_h_results.items():
    print(f"{metric:25}: {value:.4f}")

print("\nTEST METRIC")
for metric, value in test_h_results.items():
    print(f"{metric:25}: {value:.4f}")

# Sample for verification
print("\nSample Predictions (Top 10 Prediction):")
print(test_final[['FullName', 'Position', 'h_avg_pos', 'pred_top10']].head(10))


--- First Heuristic Baseline: 3-Race Rolling Average ---

TRAINING METRIC
F1 (Macro)               : 0.7689
Accuracy                 : 0.7699
ROC-AUC                  : 0.8342
PR-AUC (Avg Precision)   : 0.8229

TEST METRIC
F1 (Macro)               : 0.7689
Accuracy                 : 0.7700
ROC-AUC                  : 0.8858
PR-AUC (Avg Precision)   : 0.9041

Sample Predictions (Top 10 Prediction):
                  FullName  Position  h_avg_pos  pred_top10
444        Alexander Albon       6.0  11.333333           0
468        Alexander Albon      10.0  10.333333           0
487        Alexander Albon       9.0  11.333333           0
513        Alexander Albon      15.0   8.333333           1
529        Alexander Albon      11.0  11.333333           0
454  Andrea Kimi Antonelli      16.0  10.500000           0
466  Andrea Kimi Antonelli       8.0  16.000000           0
448           Carlos Sainz      10.0   5.666667           1
473           Carlos Sainz      15.0   8.666667           1

### Heuristic 2: "**If you drive for Red Bull, Ferrari, or McLaren, you will almost always be in the Top 10, regardless of your last three races.**"

- Leakage Prevention: Using grid position at qualifying, to predict the grid position at the race (temporal safe-guard).
- Type of baseline: Constructor (the car itself). 

In [40]:
def apply_constructor_baseline(full_df):
    """
    Heuristic 2: Constructor-Based Prediction (Expanding Window).
    Ensures that a prediction for 'Race 10' only uses averages from 'Races 1-9'.
    """
    # 1. Ensure chronological order
    df = full_df.sort_values(['Year', 'Round', 'TeamName']).copy()
    
    # 2. Calculate the 'Expanding Mean' per Team
    # .shift(1) is the "Firewall": it ensures we don't include the current race in the average
    df['team_avg_to_date'] = df.groupby('TeamName')['Position'].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    
    # 3. Handle the "First Race" problem
    # If a team has no history yet, we assume the grid midpoint (10.5)
    df['team_avg_to_date'] = df['team_avg_to_date'].fillna(10.5)
    
    # 4. Binary Classification: If team avg <= 10, predict 1 (Top 10)
    df['pred_constructor'] = (df['team_avg_to_date'] <= 10).astype(int)
    
    # 5. Probability Score (for ROC-AUC): Inverted team average
    df['c_score'] = 1 / df['team_avg_to_date']
    
    return df

# --- Execution ---
# Run this on your combined dataframe BEFORE splitting into Train/Test
processed_df = apply_constructor_baseline(full_df)

# Now split them for scoring
train_final = processed_df[processed_df['Year'] == 2024].copy()
test_final = processed_df[processed_df['Year'] == 2025].copy()

# Score the Training Set (This will now be HONEST)
train_c_results = get_scores(train_final['target_top10'], 
                             train_final['pred_constructor'], 
                             train_final['c_score'])

# Score the Test Set (This is what goes in your Comparison Table)
test_c_results = get_scores(test_final['target_top10'], 
                            test_final['pred_constructor'], 
                            test_final['c_score'])

print("\n--- Baseline 2: Constructor Tier ---")
print("\nTEST METRIC")
for metric, value in test_c_results.items():
    print(f"{metric:25}: {value:.4f}")

print("\nTRAIN METRIC")
for metric, value in train_c_results.items():
    print(f"{metric:25}: {value:.4f}")


--- Baseline 2: Constructor Tier ---

TEST METRIC
F1 (Macro)               : 0.8197
Accuracy                 : 0.8200
ROC-AUC                  : 0.8280
PR-AUC (Avg Precision)   : 0.8691

TRAIN METRIC
F1 (Macro)               : 0.7823
Accuracy                 : 0.7836
ROC-AUC                  : 0.8349
PR-AUC (Avg Precision)   : 0.8054


## CELL 2.2: MODELS

### 2.2.1: Scaler

- First we scale the data (based on train data)

In [45]:
# 1. DEFINE COLUMN GROUPS BASED ON YOUR IMAGE
# These are the numbers that need to be "Standardized" (Mean=0, Std=1)
numeric_features = [
    'Round', 'BestLapTime_s', 'LapTimeDelta', 'QStageReached', 
    'GridPosition', 'driver_form', 'team_strength', 'avg_delta_season'
]

# These are the words that need to be turned into 0/1 binary columns
categorical_features = ['TeamName', 'Circuit']

# This is what the model is trying to predict
target = 'target_top10'

# 2. TEMPORAL SPLIT
train_mask = full_df['Year'] == 2024
test_mask = full_df['Year'] == 2025

# 3. ONE-HOT ENCODING (Categoricals)
# We do this first so both years have the exact same columns
df_encoded = pd.get_dummies(full_df, columns=categorical_features, drop_first=True)

# Identify the names of the new 0/1 columns created from TeamName and Circuit
encoded_cols = [c for c in df_encoded.columns if any(cat in c for cat in categorical_features)]

# 4. SCALING (Numerics)
scaler = StandardScaler()

# Fit ONLY on 2024 to prevent data leakage
X_train_num = scaler.fit_transform(df_encoded.loc[train_mask, numeric_features])
X_test_num = scaler.transform(df_encoded.loc[test_mask, numeric_features])

# 5. CONCATENATE NUMERIC + CATEGORICAL
# We use np.hstack to combine the scaled numbers and the binary categories into one matrix
X_train = np.hstack([X_train_num, df_encoded.loc[train_mask, encoded_cols].values])
X_test = np.hstack([X_test_num, df_encoded.loc[test_mask, encoded_cols].values])

# 6. EXTRACT LABELS
y_train = full_df.loc[train_mask, target].values
y_test = full_df.loc[test_mask, target].values

print("--- FEATURE SUMMARY ---")
print(f"Total Features used: {X_train.shape[1]}")
print(f"Numeric Scaled: {len(numeric_features)}")
print(f"Categorical (Encoded): {len(encoded_cols)}")

--- FEATURE SUMMARY ---
Total Features used: 39
Numeric Scaled: 8
Categorical (Encoded): 31
